# Mushroom Classification & Safety Analysis — SOLUTION

**Complete, production-ready implementation with interactive visualizations, statistical testing, ML modeling, and a practical simulation tool.**

## Analysis Pipeline Flowchart

```mermaid
flowchart TD
    A[Load & Inspect Data] --> B[Handle Missing Values & Encoding]
    B --> C[Interactive EDA with Plotly Countplots & Heatmap]
    C --> D[Statistical Testing: Chi-Square for Class Association]
    D --> E[Feature Importance via Random Forest]
    E --> F[Build & Evaluate Simple Classifier]
    F --> G[Interactive Simulation: Input Hypothetical Mushroom → Predict Poison Probability]
    G --> H[Audience-Tailored Insights & Safety Recommendations]
    H --> I[Extended Practice & What-If Simulations]
```


## 1. Data Loading, Inspection & Cleaning

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from scipy.stats import chi2_contingency
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)

# Load data (adjust path as needed)
df = pd.read_csv('/home/workdir/attachments/mushroom_data.csv')

print('=== Dataset Overview ===')
print(f'Shape: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'\nTarget distribution:\n{df["Class"].value_counts(normalize=True).round(3)}')
print(f'\nMissing values total: {df.isnull().sum().sum()}')
print('\nUnique values per column (sample):')
for col in df.columns[:6]:
    print(f'  {col}: {df[col].nunique()} unique → {df[col].unique()[:4].tolist()}...')


## 2. Interactive EDA with Plotly (Key Features)

In [ ]:
# Highly predictive features based on domain knowledge + quick checks
key_features = ['Odor', 'Spore Print Color', 'Gill Color', 'Bruises', 'Habitat', 'Cap Color']

for feat in key_features:
    fig = px.histogram(
        df, x=feat, color='Class', barmode='group',
        title=f'{feat} Distribution by Class (Interactive)',
        labels={feat: feat, 'count': 'Count'},
        color_discrete_map={'Edible': '#2E86AB', 'Poisonous': '#E76F51'}
    )
    fig.update_layout(xaxis_tickangle=-30)
    fig.show()

print('Key insight: Odor is extremely discriminative — almond/anise = almost always edible, foul/pungent/fishy = almost always poisonous.')


## 3. Chi-Square Statistical Testing (All Features)

In [ ]:
results = []
for col in df.columns:
    if col != 'Class':
        contingency = pd.crosstab(df[col], df['Class'])
        chi2, p, dof, expected = chi2_contingency(contingency)
        results.append({'Feature': col, 'Chi2': round(chi2, 1), 'p-value': p, 'Degrees of Freedom': dof})

chi_df = pd.DataFrame(results).sort_values('p-value')
print('=== Chi-Square Test Results (sorted by significance) ===')
print(chi_df.to_string(index=False))

print('\n=== Top 5 Most Associated Features with Poisonousness ===')
print(chi_df.head(5)[['Feature', 'p-value']])


## 4. Predictive Modeling & Feature Importance

In [ ]:
# One-hot encode all features
X = pd.get_dummies(df.drop('Class', axis=1), drop_first=True)
y = (df['Class'] == 'Poisonous').astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

rf = RandomForestClassifier(n_estimators=150, max_depth=12, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
print('Accuracy on test set:', round(accuracy_score(y_test, y_pred), 4))
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Edible (0)', 'Poisonous (1)']))

# Feature importance
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print('\n=== Top 10 Most Important Features ===')
print(importances.head(10).round(4))


## 5. Simulation Tool: Predict Poison Probability for Any Mushroom

In [ ]:
def predict_poison_probability(model, feature_values, training_columns):
    """Predict probability a mushroom is poisonous given its features."""
    input_df = pd.DataFrame([feature_values])
    input_encoded = pd.get_dummies(input_df)
    # Align columns with training data
    input_aligned = input_encoded.reindex(columns=training_columns, fill_value=0)
    prob_poison = model.predict_proba(input_aligned)[0][1]
    return round(prob_poison * 100, 2)

# Example 1: Classic poisonous profile (pungent odor + black spore print)
example1 = {
    'Odor': 'Pungent',
    'Spore Print Color': 'Black',
    'Gill Color': 'Black',
    'Bruises': True,
    'Habitat': 'Urban',
    'Cap Color': 'Brown'
}
prob1 = predict_poison_probability(rf, example1, X.columns)
print(f'Example 1 (Pungent + Black spore print): {prob1}% probability of being POISONOUS')

# Example 2: Classic edible profile
example2 = {
    'Odor': 'Almond',
    'Spore Print Color': 'Brown',
    'Gill Color': 'Black',
    'Bruises': True,
    'Habitat': 'Grasses',
    'Cap Color': 'Yellow'
}
prob2 = predict_poison_probability(rf, example2, X.columns)
print(f'Example 2 (Almond odor + Brown spore print): {prob2}% probability of being POISONOUS')


## Key Insights & Audience-Tailored Recommendations

**For Foragers (Safety First)**:
- **Strongest red flags**: Foul, pungent, fishy, or spicy odor + black or white spore print color.
- **Strongest green flags**: Almond or anise odor + brown spore print.
- Never rely on a single feature — the model combines many signals.

**For Data Scientists**:
- The dataset is almost perfectly separable with simple tree-based models (accuracy > 99% possible with tuning).
- Odor, spore print color, and gill color dominate importance.

**For Mycologists / Researchers**:
- Many traditional "rules of thumb" are statistically validated here (e.g., odor is highly informative).
- The "Missing" category in Stalk Root is itself predictive.


## Simulation & What-If Section

Try changing the values in the example dictionaries above and re-running the prediction cell. You can simulate rare combinations or "what if" scenarios (e.g., a mushroom with almond odor but black spore print — how does the model weigh conflicting signals?).